# LoRA Compress - Q4 Quantized Model Adaptive Autoresearch

This version loads a Q4 quantized model and trains LoRA to match the **quantized weights**.

**Hypothesis:** Since Q4 weights already have quantization error, they may be more
compressible with low-rank approximation (lower intrinsic rank).

**Features:**
- Uses bitsandbytes for Q4 quantization
- **Configurable analysis rank** - test ranks 16, 32, 64 to find optimal
- Compare Q4 vs FP16 compression difficulty at multiple ranks
- Same OAT (Ordered Adaptive Testing) strategy

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/LoRA_Compress'
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/databases', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/results', exist_ok=True)

print(f'✅ Drive mounted: {DRIVE_BASE}')

In [ ]:
%cd /content
!git clone https://github.com/qades/loracompress.git 2>/dev/null || true
%cd loracompress

# Install dependencies including bitsandbytes for quantization
!pip install -q transformers torch optuna tqdm bitsandbytes accelerate

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
import random
import numpy as np
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

print(f'🔥 PyTorch: {torch.__version__}')
print(f'🔥 GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## Step 4: Load Q4 Quantized Model

In [ ]:
MODEL_NAME = 'HuggingFaceTB/SmolLM2-135M'

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print('Loading Q4 quantized model...')
model_q4 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

print(f'\n✅ Q4 Model loaded')

## Step 5: Also Load FP16 Model for Comparison

In [ ]:
print('Loading FP16 model for comparison...')

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='cpu',
    trust_remote_code=True,
)

print('✅ FP16 model loaded (on CPU)')

## Step 6: Layer Browser with Configurable Rank Analysis

**NEW**: Test multiple ranks to see how low you can go with Q4!

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURABLE ANALYSIS RANKS
# ═══════════════════════════════════════════════════════════════

# Which ranks to analyze? Test low ranks to see Q4 advantage!
ANALYSIS_RANKS = [16, 32, 64]  # Add 8 or 128 as needed

# Primary rank for comparison (shown in main table)
PRIMARY_RANK = 32  # Set to 16 or 32 to see if Q4 gets 100% at low ranks

print(f'Analysis ranks: {ANALYSIS_RANKS}')
print(f'Primary comparison rank: {PRIMARY_RANK}\n')

def get_layer_svd_stats(weight, ranks=[16, 32, 64]):
    """Compute SVD stats for multiple ranks."""
    w = weight.float()
    try:
        U, S, Vh = torch.linalg.svd(w, full_matrices=False)
        total = (S ** 2).sum()
        
        results = {}
        for r in ranks:
            if r <= len(S):
                # Variance explained
                var = (S[:r] ** 2).sum() / total * 100
                
                # L1 error for this rank
                S_r = torch.zeros_like(S)
                S_r[:r] = S[:r]
                W_r = U @ torch.diag(S_r) @ Vh
                l1_err = torch.mean(torch.abs(W_r - w)).item() / torch.mean(torch.abs(w)).item() * 100
                
                results[r] = {'var': var.item(), 'l1_err': l1_err}
            else:
                results[r] = {'var': 100.0, 'l1_err': 0.0}
        
        return results
    except:
        return {r: {'var': 0, 'l1_err': 100} for r in ranks}

print('Analyzing layers: Q4 vs FP16 compressibility\n')
print(f'Rank {PRIMARY_RANK} Comparison:')
print('=' * 110)

# Build header
header = f"{'Layer':<40}"
for r in ANALYSIS_RANKS:
    header += f" {'Q4 R'+str(r):>7} {'FP16 R'+str(r):>9}"
header += f" {'Q4 Easier':>10}"
print(header)
print('=' * 110)

layer_comparisons = []

for (name_q4, param_q4), (name_fp16, param_fp16) in zip(
    model_q4.named_parameters(), model_fp16.named_parameters()
):
    if len(param_q4.shape) != 2 or param_q4.shape[0] < 100:
        continue
    
    w_q4 = param_q4.data.cpu().float()
    w_fp16 = param_fp16.data.cpu().float()
    
    stats_q4 = get_layer_svd_stats(w_q4, ANALYSIS_RANKS)
    stats_fp16 = get_layer_svd_stats(w_fp16, ANALYSIS_RANKS)
    
    # Q4 is easier if at PRIMARY_RANK it has higher variance or lower error
    q4_easier = (stats_q4[PRIMARY_RANK]['var'] > stats_fp16[PRIMARY_RANK]['var'] or
                 stats_q4[PRIMARY_RANK]['l1_err'] < stats_fp16[PRIMARY_RANK]['l1_err'])
    
    layer_comparisons.append({
        'name': name_q4,
        'shape': tuple(param_q4.shape),
        'stats_q4': stats_q4,
        'stats_fp16': stats_fp16,
        'q4_easier': q4_easier,
        'w_q4': w_q4,
        'w_fp16': w_fp16,
    })
    
    # Print first 25 layers
    if len(layer_comparisons) <= 25:
        easier_mark = '✓' if q4_easier else '✗'
        parts = name_q4.split('.')
        name_short = '.'.join(parts[-2:]) if len(parts) >= 2 else name_q4[:35]
        
        line = f"{name_short:<40}"
        for r in ANALYSIS_RANKS:
            line += f" {stats_q4[r]['var']:>7.1f} {stats_fp16[r]['var']:>9.1f}"
        line += f" {easier_mark:>10}"
        print(line)

print('\n' + '=' * 110)

# Summary
q4_easier_count = sum(1 for l in layer_comparisons if l['q4_easier'])
print(f"\n📊 Summary at rank {PRIMARY_RANK}:")
print(f"   Q4 easier to compress: {q4_easier_count}/{len(layer_comparisons)} layers ({q4_easier_count/len(layer_comparisons)*100:.1f}%)")

# Show how many layers get >95% variance at each rank
print(f"\n📈 Layers with >95% variance captured:")
for r in ANALYSIS_RANKS:
    q4_good = sum(1 for l in layer_comparisons if l['stats_q4'][r]['var'] > 95)
    fp16_good = sum(1 for l in layer_comparisons if l['stats_fp16'][r]['var'] > 95)
    print(f"   Rank {r:3d}: Q4={q4_good:2d}/{len(layer_comparisons)} ({q4_good/len(layer_comparisons)*100:.0f}%), "
          f"FP16={fp16_good:2d}/{len(layer_comparisons)} ({fp16_good/len(layer_comparisons)*100:.0f}%)")

# Find best Q4 layer at lowest rank possible
for test_rank in sorted(ANALYSIS_RANKS):
    good_q4 = [l for l in layer_comparisons if l['stats_q4'][test_rank]['var'] > 95 and l['q4_easier']]
    if good_q4:
        best = max(good_q4, key=lambda x: x['stats_q4'][test_rank]['var'] - x['stats_fp16'][test_rank]['var'])
        print(f"\n🏆 Best at rank {test_rank}: {best['name']}")
        print(f"   Q4 var: {best['stats_q4'][test_rank]['var']:.1f}%, FP16 var: {best['stats_fp16'][test_rank]['var']:.1f}%")
        break

## Step 7: Select Target Layer

In [ ]:
# ═══════════════════════════════════════════════════════════════
# LAYER SELECTION
# ═══════════════════════════════════════════════════════════════

LAYER_INDEX = None
LAYER_PATTERN = 'layers.15'
AUTO_SELECT_BEST_Q4 = True

# Target rank for selection (can be different from analysis rank)
TARGET_RANK_FOR_SELECTION = 32  # Try 16 if many layers show >95% at rank 16!

def select_layer(comparisons, index=None, pattern=None, auto_best=False, target_rank=32):
    if auto_best:
        good_layers = [l for l in comparisons if l['stats_q4'][target_rank]['var'] > 90]
        if good_layers:
            return max(good_layers, key=lambda x: x['stats_q4'][target_rank]['var'] - x['stats_fp16'][target_rank]['var'])
    
    if index is not None and index < len(comparisons):
        return comparisons[index]
    
    if pattern:
        matches = [l for l in comparisons if pattern in l['name']]
        if matches:
            easier = [m for m in matches if m['q4_easier']]
            if easier:
                return max(easier, key=lambda x: x['stats_q4'][target_rank]['var'])
            return matches[0]
    
    easier_mid = [l for l in comparisons if l['q4_easier'] and 'layers.10' in l['name'] or 'layers.15' in l['name']]
    if easier_mid:
        return easier_mid[len(easier_mid)//2]
    return comparisons[len(comparisons)//2]

selected = select_layer(layer_comparisons, LAYER_INDEX, LAYER_PATTERN, AUTO_SELECT_BEST_Q4, TARGET_RANK_FOR_SELECTION)

print(f"✅ Selected: {selected['name']}")
print(f"   Shape: {selected['shape']}\n")

print(f"📊 Q4 vs FP16 Comparison at multiple ranks:")
for r in ANALYSIS_RANKS:
    q4v = selected['stats_q4'][r]['var']
    fp16v = selected['stats_fp16'][r]['var']
    q4e = selected['stats_q4'][r]['l1_err']
    fp16e = selected['stats_fp16'][r]['l1_err']
    print(f"   Rank {r:3d}: Q4 var={q4v:5.1f}% FP16 var={fp16v:5.1f}% | Q4 L1={q4e:5.1f}% FP16 L1={fp16e:5.1f}%")

target_weight_q4 = selected['w_q4']
target_weight_fp16 = selected['w_fp16']

print(f"\n📐 Using Q4 weights: {target_weight_q4.shape}")

## Step 8: Training Functions

In [ ]:
class AdaptiveExplorer:
    def __init__(self, lr_min=0.0005, lr_max=0.8, rank_min=8, rank_max=256, aggressive_ratio=0.3):
        self.lr_min = lr_min
        self.lr_max = lr_max
        self.rank_min = rank_min
        self.rank_max = rank_max
        self.aggressive_ratio = aggressive_ratio
        self.lr_history = []
        self.rank_history = []
        self.lr_upper_boundary = None
        self.lr_optimal_region = None
        self.rank_effectiveness = {}
        self.oat_phase = 'explore_lr_up'
        self.oat_test_values = [0.01, 0.02, 0.04, 0.08, 0.16, 0.32, 0.64]
        self.oat_test_idx = 0
        self.oat_best_lr = None

    def sample_lr_two_region(self, trial):
        if random.random() < self.aggressive_ratio:
            return trial.suggest_float('lr', 0.1, self.lr_max, log=True)
        return trial.suggest_float('lr', self.lr_min, 0.1, log=True)

    def sample_lr_adaptive(self, trial):
        if self.lr_optimal_region and random.random() < 0.6:
            low, high = self.lr_optimal_region
            return trial.suggest_float('lr', max(self.lr_min, low * 0.8),
                                        min(self.lr_max, high * 1.2), log=True)
        return self.sample_lr_two_region(trial)

    def sample_rank_adaptive(self, trial):
        if self.rank_effectiveness and random.random() < 0.5:
            sorted_ranks = sorted(self.rank_effectiveness.items(), key=lambda x: x[1])
            good_ranks = [r for r, err in sorted_ranks[:3] if err < 50]
            if good_ranks:
                base_rank = random.choice(good_ranks)
                low = max(self.rank_min, int(base_rank * 0.75))
                high = min(self.rank_max, int(base_rank * 1.25))
                return trial.suggest_int('rank', low, high, log=True)
        return trial.suggest_int('rank', self.rank_min, self.rank_max, log=True)

    def update_from_result(self, config, error):
        lr = config.get('lr')
        rank = config.get('rank')
        if lr:
            self.lr_history.append((lr, error))
        if rank:
            self.rank_history.append((rank, error))
            self.rank_effectiveness[rank] = error
        if len(self.lr_history) >= 5:
            sorted_lrs = sorted(self.lr_history, key=lambda x: x[0])
            best_err = min(e for _, e in sorted_lrs)
            good_lrs = [lr for lr, err in sorted_lrs if err < best_err * 1.5]
            if good_lrs:
                self.lr_optimal_region = (min(good_lrs), max(good_lrs))

    def get_oat_suggestion(self, trial_number):
        if trial_number == 0:
            return {'rank': 32, 'lr': 0.01, 'epochs': 500}
        if self.oat_phase == 'explore_lr_up':
            if len(self.lr_history) >= 2 and self.lr_history[-1][1] > self.lr_history[-2][1] * 1.3:
                self.lr_upper_boundary = self.lr_history[-1][0]
                self.oat_best_lr = self.lr_history[-2][0]
                self.oat_phase = 'refine_lr_down'
                self.oat_test_values = [self.oat_best_lr * (0.95 ** i) for i in range(1, 10)]
                self.oat_test_idx = 0
                print(f'  [OAT] Upper boundary at {self.lr_upper_boundary:.4f}')
            if self.oat_test_idx < len(self.oat_test_values):
                lr = self.oat_test_values[self.oat_test_idx]
                self.oat_test_idx += 1
                return {'rank': 32, 'lr': lr, 'epochs': 500}
        if self.oat_phase == 'refine_lr_down':
            if len(self.lr_history) >= 2 and self.lr_history[-1][1] > self.lr_history[-2][1]:
                self.oat_best_lr = self.lr_history[-2][0]
                self.oat_phase = 'explore_rank'
                self.oat_test_values = [8, 12, 16, 24, 32, 48, 64, 96, 128]
                self.oat_test_idx = 0
                print(f'  [OAT] Best LR: {self.oat_best_lr:.4f}, exploring ranks')
            if self.oat_test_idx < len(self.oat_test_values):
                lr = self.oat_test_values[self.oat_test_idx]
                self.oat_test_idx += 1
                return {'rank': 32, 'lr': lr, 'epochs': 500}
        if self.oat_phase == 'explore_rank':
            if self.oat_test_idx < len(self.oat_test_values):
                rank = self.oat_test_values[self.oat_test_idx]
                self.oat_test_idx += 1
                return {'rank': rank, 'lr': self.oat_best_lr or 0.01, 'epochs': 500}
            self.oat_phase = 'exploit'
            print('  [OAT] Switching to exploitation')
        return None

def compute_l1_error(W_approx, target):
    l1_error = torch.mean(torch.abs(W_approx - target)).item()
    mean_abs_target = torch.mean(torch.abs(target)).item()
    return (l1_error / mean_abs_target * 100) if mean_abs_target > 0 else float('inf')

def train_lora(target_weight, rank, lr, epochs, device='cuda', scheduler_type=None):
    d, k = target_weight.shape
    target = target_weight.float().to(device)
    A = nn.Parameter(torch.randn(rank, k, device=device) * 0.01)
    B = nn.Parameter(torch.randn(d, rank, device=device) * 0.01)
    optimizer = torch.optim.AdamW([A, B], lr=lr)
    
    scheduler = None
    if scheduler_type == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr*0.01)
    elif scheduler_type == 'linear':
        scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.01, total_iters=epochs)
    
    best_loss = float('inf')
    best_A, best_B = None, None
    epochs_no_improve = 0
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        W_approx = torch.matmul(B, A)
        loss = F.mse_loss(W_approx, target)
        if not torch.isfinite(loss):
            return float('inf'), 0
        loss.backward()
        optimizer.step()
        if scheduler:
            scheduler.step()
        current = loss.item()
        if current < best_loss - 1e-9:
            best_loss = current
            best_A = A.detach().clone()
            best_B = B.detach().clone()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epoch >= 200 and epochs_no_improve >= 100:
            break
    if best_A is None:
        return float('inf'), 0
    with torch.no_grad():
        W_best = torch.matmul(best_B, best_A)
        l1_error = compute_l1_error(W_best, target)
    return l1_error, epoch + 1

print('✅ Functions loaded')

## Step 9: Quick Test Q4 vs FP16

In [ ]:
print('Quick comparison: Training on Q4 vs FP16 weights\n')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

test_ranks = [16, 32, 64]  # Same as ANALYSIS_RANKS

for test_rank in test_ranks:
    print(f'\nRank {test_rank}:')
    
    err_q4, epochs_q4 = train_lora(target_weight_q4, test_rank, 0.01, 500, device)
    print(f'  Q4:   {err_q4:.2f}% L1 error ({epochs_q4} epochs)')
    
    err_fp16, epochs_fp16 = train_lora(target_weight_fp16, test_rank, 0.01, 500, device)
    print(f'  FP16: {err_fp16:.2f}% L1 error ({epochs_fp16} epochs)')
    
    if err_q4 < err_fp16:
        print(f'  ✅ Q4 better by {err_fp16 - err_q4:.2f}%')
    else:
        print(f'  ❌ FP16 better by {err_q4 - err_fp16:.2f}%')
    
    # Compare to SVD theoretical
    svd_q4 = selected['stats_q4'][test_rank]['l1_err']
    svd_fp16 = selected['stats_fp16'][test_rank]['l1_err']
    print(f'  (SVD theoretical: Q4={svd_q4:.1f}%, FP16={svd_fp16:.1f}%)')

## Step 10: Configuration & Run

In [ ]:
TARGET_QUALITY = 5.0
N_TRIALS = 50
EXPLORATION_MODE = 'oat'

# Set rank range based on SVD analysis
# If rank 16 gives >95% variance, start lower!
best_var_at_16 = selected['stats_q4'][16]['var'] if 16 in selected['stats_q4'] else 0
best_var_at_32 = selected['stats_q4'][32]['var'] if 32 in selected['stats_q4'] else 0

if best_var_at_16 > 95:
    RANK_MIN, RANK_MAX = 8, 64
    print(f'Excellent Q4 compressibility! Rank range: {RANK_MIN}-{RANK_MAX} (rank-16 variance: {best_var_at_16:.1f}%)')
elif best_var_at_32 > 95:
    RANK_MIN, RANK_MAX = 16, 128
    print(f'Good Q4 compressibility. Rank range: {RANK_MIN}-{RANK_MAX} (rank-32 variance: {best_var_at_32:.1f}%)')
else:
    RANK_MIN, RANK_MAX = 32, 256
    print(f'Standard compressibility. Rank range: {RANK_MIN}-{RANK_MAX}')

explorer = AdaptiveExplorer(lr_min=0.0005, lr_max=0.8, rank_min=RANK_MIN, rank_max=RANK_MAX)
target_weight_gpu = target_weight_q4.to(device)

def objective(trial):
    trial_num = len(explorer.lr_history)
    if EXPLORATION_MODE == 'oat':
        oat_config = explorer.get_oat_suggestion(trial_num)
        if oat_config:
            rank = oat_config['rank']
            lr = oat_config['lr']
            epochs = oat_config['epochs']
            scheduler = None
            print(f'  [OAT {explorer.oat_phase}] r={rank}, lr={lr:.4f}')
        else:
            rank = explorer.sample_rank_adaptive(trial)
            lr = explorer.sample_lr_adaptive(trial)
            epochs = trial.suggest_int('epochs', 200, 2000, log=True)
            scheduler = trial.suggest_categorical('scheduler', [None, 'cosine', 'linear'])
    else:
        rank = trial.suggest_int('rank', RANK_MIN, RANK_MAX, log=True)
        lr = explorer.sample_lr_two_region(trial)
        epochs = trial.suggest_int('epochs', 200, 2000, log=True)
        scheduler = None
    
    error, actual_epochs = train_lora(target_weight_gpu, rank, lr, epochs, device, scheduler)
    explorer.update_from_result({'rank': rank, 'lr': lr, 'epochs': epochs}, error)
    
    d, k = target_weight_q4.shape
    compression = (d * k) / (rank * (d + k))
    score = -compression * 10 if error <= TARGET_QUALITY else 1000 + (error - TARGET_QUALITY) * 100
    
    trial.set_user_attr('error', error)
    trial.set_user_attr('compression', compression)
    trial.set_user_attr('actual_epochs', actual_epochs)
    return score

print('✅ Ready')

In [ ]:
import os

study_name = f'q4_{EXPLORATION_MODE}_{TARGET_QUALITY}pct_{LAYER_PATTERN.replace(".", "_")}_r{RANK_MIN}-{RANK_MAX}'
db_file = f'{DRIVE_BASE}/databases/{study_name}.db'
os.makedirs(os.path.dirname(db_file), exist_ok=True)

study = optuna.create_study(
    study_name=study_name,
    storage=f'sqlite:///{db_file}',
    direction='minimize',
    load_if_exists=True
)

print(f'\n🚀 Running {N_TRIALS} trials on Q4 weights')
print(f'   Target: <{TARGET_QUALITY}% error')
print(f'   Layer: {selected["name"]}')
print(f'   Rank range: {RANK_MIN}-{RANK_MAX}\n')

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ Best: {study.best_trial.user_attrs["error"]:.2f}% error')
print(f'   Config: rank={study.best_params.get("rank")}, lr={study.best_params.get("lr"):.4f}')
print(f'   Compression: {study.best_trial.user_attrs["compression"]:.1f}x')

In [ ]:
# Summary
results = []
for trial in study.trials:
    if trial.state == optuna.trial.TrialState.COMPLETE:
        results.append({
            'rank': trial.params.get('rank'),
            'lr': trial.params.get('lr'),
            'error': trial.user_attrs.get('error'),
            'compression': trial.user_attrs.get('compression'),
        })

results.sort(key=lambda x: x['error'])

print('\n🏆 TOP 10 RESULTS - Q4 Model')
print(f"{'Rank':>6} {'LR':>10} {'Error%':>10} {'Compr':>10}")
print('-' * 50)
for r in results[:10]:
    m = '✓' if r['error'] <= TARGET_QUALITY else '✗'
    print(f"{r['rank']:>6} {r['lr']:>10.4f} {r['error']:>9.2f}{m} {r['compression']:>9.1f}x")

# Save
import json
output = {
    'model': MODEL_NAME,
    'quantization': 'Q4_NF4',
    'layer': selected['name'],
    'target_quality': TARGET_QUALITY,
    'best': {
        'rank': study.best_params.get('rank'),
        'lr': study.best_params.get('lr'),
        'error': study.best_trial.user_attrs.get('error'),
    },
    'q4_vs_fp16_svd': {str(r): selected['stats_q4'][r]['var'] - selected['stats_fp16'][r]['var'] for r in ANALYSIS_RANKS},
}
with open(f'{DRIVE_BASE}/results/{study_name}.json', 'w') as f:
    json.dump(output, f, indent=2)
print(f'\n✅ Saved to Drive')